# 分词（Tokenization）与词嵌入

文本是人类语言，神经网络只懂数字。**分词**就是把文字变成数字的第一道工序，也是最容易被忽视但影响深远的一步。

本章覆盖：
- 文本表示的三个流派（字符级 / 词级 / 子词级）及其取舍
- BPE 算法原理 + 从零用 Python 实现，附可视化
- 字节级 BPE：为什么 GPT 能处理任意语言和 emoji
- tiktoken 实战：token 计数、成本估算、上下文窗口
- 多语言的 token 效率差异（为什么中文消耗更多 token）
- 词嵌入（Embedding）：从 token ID 到高维向量，可视化语义空间

**依赖：** `numpy`, `matplotlib`, `tiktoken`（`pip install tiktoken`），其余为标准库

## 第一章：从文字到数字——三种方案的演进

神经网络需要数字输入。如何把文本变成数字，经历了三代方案。

### 方案一：字符级（Character-level）

每个字符作为一个独立单元，直接用 ASCII/Unicode 码：

```
"cat"   ->  [99, 97, 116]
"Hello" ->  [72, 101, 108, 108, 111]
```

**优点：** 词表极小（256 字节），天然处理拼写错误和新词。
**缺点：** 序列太长（1000 词的文章约 5000+ 字符），单字符本身没有语义，模型需要从字符堆里「学出」词语含义，非常困难。

---

### 方案二：词级（Word-level）

整个词作为最小单元：

```
"the cat sat"  ->  [词典中: the=1, cat=2, sat=3]
```

**优点：** 每个 token 直接有语义，序列长度适中。
**缺点：** 词表巨大（英文 50 万+词），新词（ChatGPT、RLHF）无法处理（OOV 问题），不同语言分词规则不一样。

---

### 方案三：子词级（Subword）—— 现代 LLM 的选择

在字符和词之间取平衡：

```
"tokenization" ->  ["token", "ization"]
"ChatGPT"      ->  ["Chat", "G", "PT"]
"unhappiness"  ->  ["un", "happiness"]
```

**优势：**
- 常见词作为整体 token，保留语义
- 罕见词拆成常见子词，解决 OOV 问题
- 词表大小可控（GPT-4 使用 10 万词表）

In [ ]:
import re
import collections
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 三种方案的序列长度对比
text_sample = "The tokenization process converts text into numbers for neural networks to process efficiently."

char_tokens = list(text_sample)
word_tokens = text_sample.split()
subword_len = int(len(word_tokens) * 1.3)

print(f"原始文本: {text_sample!r}")
print(f"  字符级: {len(char_tokens)} tokens")
print(f"  词级:   {len(word_tokens)} tokens")
print(f"  子词级: ~{subword_len} tokens")

fig, ax = plt.subplots(figsize=(10, 5))
methods = ['字符级\n(Character-level)', '词级\n(Word-level)', '子词级\n(Subword, GPT)']
lengths = [len(char_tokens), len(word_tokens), subword_len]
colors  = ['#e74c3c', '#f39c12', '#27ae60']
bars    = ax.bar(methods, lengths, color=colors, width=0.5, edgecolor='white', linewidth=1.5)

for bar, l in zip(bars, lengths):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(l), ha='center', va='bottom', fontsize=14, fontweight='bold')

problems  = ['序列太长\n训练困难', '新词无法处理\n词表过大', '最优平衡点\n主流方案']
prob_cols = ['#e74c3c', '#f39c12', '#27ae60']
for i, (txt, c) in enumerate(zip(problems, prob_cols)):
    ax.text(i, lengths[i] * 0.45, txt, ha='center', va='center', fontsize=9,
            color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=c, alpha=0.8))

ax.set_ylabel('Token 数量 (序列长度)', fontsize=12)
ax.set_title('同一段英文文本：三种表示方案的序列长度对比', fontsize=13)
ax.set_ylim(0, max(lengths) * 1.3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('tokenization_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 第二章：BPE 算法—— 子词是怎么学出来的？

BPE（Byte Pair Encoding，字节对编码）最早是数据压缩算法，2016 年被引入 NLP。

### 直觉理解

**核心思想：反复把语料中最常见的相邻 token 对合并成新 token，重复 N 次。**

好比文字压缩：如果 `th` 总是连在一起出现，就创造一个新符号 `θ` 代替 `th`，节省空间。

### 算法步骤图解

**训练语料（4 个词及其频率）：**

| 词 | 频率 | 初始 token 序列（词尾加 `</w>` 区分边界） |
|---|------|------|
| low | 5 | `l` `o` `w` `</w>` |
| lower | 2 | `l` `o` `w` `e` `r` `</w>` |
| newest | 6 | `n` `e` `w` `e` `s` `t` `</w>` |
| widest | 3 | `w` `i` `d` `e` `s` `t` `</w>` |

**迭代 1：统计所有相邻对频率**
```
(e, s):  newest(6) + widest(3) = 9 次  <- 最高频
(s, t):  newest(6) + widest(3) = 9 次  <- 并列
(e, st): 还没出现（上轮尚未合并）
...
```
合并 `e` + `s` -> `es`，更新词表。

**迭代 2：重新统计**
```
(es, t): newest(6) + widest(3) = 9 次  <- 新的最高
...
```
合并 `es` + `t` -> `est`

如此反复，直到达到目标词表大小（通常几万到几十万次合并）。

> **为什么词尾加 `</w>`？**
> 区分「low」中的 `ow` 和独立词「ow」。有了 `</w>` 边界标记，合并出的子词不会跨词边界。

In [ ]:
def get_vocab(corpus):
    # 将语料转为初始词表：每个字符独立，词尾加 </w>
    vocab = {}
    for word, freq in corpus.items():
        chars = list(word) + ['</w>']
        vocab[' '.join(chars)] = freq
    return vocab

def get_stats(vocab):
    # 统计所有相邻 token 对的总频率
    pairs = collections.Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    # 在整个词表中合并指定的 token 对
    bigram      = ' '.join(pair)
    replacement = ''.join(pair)
    return {word.replace(bigram, replacement): freq for word, freq in vocab.items()}

# ==================  BPE 完整训练演示  ==================
corpus = {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}
vocab  = get_vocab(corpus)

print("初始词表（每个字符是独立 token）:")
for word, freq in vocab.items():
    print(f"  {word.split()}  x{freq}")

num_merges    = 10
merge_history = []

print(f"\n{'='*55}")
print("BPE 迭代合并（共 10 次）")
print(f"{'='*55}")

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best_pair  = max(pairs, key=pairs.get)
    best_count = pairs[best_pair]
    vocab      = merge_vocab(best_pair, vocab)
    new_token  = ''.join(best_pair)
    merge_history.append((i+1, best_pair, best_count, new_token))
    print(f"\n合并 #{i+1:02d}: '{best_pair[0]}' + '{best_pair[1]}' -> '{new_token}'  (频率={best_count})")
    for word, freq in vocab.items():
        print(f"  {word.split()}  x{freq}")

In [ ]:
# 可视化 BPE 合并过程
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图：每步合并的频率
merge_nums   = [m[0] for m in merge_history]
merge_counts = [m[2] for m in merge_history]
new_tokens   = [m[3] for m in merge_history]
pair_labels  = [f"'{m[1][0]}'+'{m[1][1]}'" for m in merge_history]

bars = axes[0].bar(merge_nums, merge_counts, color='#3498db', edgecolor='white', linewidth=1.2)
for bar, plabel, tok in zip(bars, pair_labels, new_tokens):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'->{tok}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                 plabel, ha='center', va='center', fontsize=7, color='white')

axes[0].set_xlabel('合并步骤编号')
axes[0].set_ylabel('被合并 token 对的出现频率')
axes[0].set_title('BPE 每步合并情况\n(柱内=合并哪两个，柱顶=合并成什么)')
axes[0].set_xticks(merge_nums)
axes[0].set_xticklabels([f'#{n}' for n in merge_nums], rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# 右图：以 "newest" 为例，展示 token 数量的变化
word_example = 'newest'
current_repr = ' '.join(list(word_example) + ['</w>'])
repr_history = [current_repr.split()]
tmp_vocab = get_vocab(corpus)

for _, best_pair, _, _ in merge_history:
    bigram      = ' '.join(best_pair)
    replacement = ''.join(best_pair)
    current_repr = current_repr.replace(bigram, replacement)
    repr_history.append(current_repr.split())

steps  = list(range(len(repr_history)))
counts = [len(r) for r in repr_history]

axes[1].plot(steps, counts, 'o-', color='#e74c3c', linewidth=2.5, markersize=10, zorder=5)
axes[1].fill_between(steps, counts, alpha=0.15, color='#e74c3c')
for s, c, r in zip(steps, counts, repr_history):
    axes[1].annotate(str(r), (s, c), textcoords="offset points",
                     xytext=(0, 10), ha='center', fontsize=7, color='#333',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='#f0f0f0', alpha=0.7))

axes[1].set_xlabel('合并步骤')
axes[1].set_ylabel('Token 数量')
axes[1].set_title(f'"{word_example}" 的 token 分解变化\n初始={counts[0]} 个字符 token，随合并逐渐减少')
axes[1].set_xticks(steps)
axes[1].set_xticklabels(['初始'] + [f'#{n}' for n in merge_nums], rotation=45)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bpe_process.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n最终词表中出现的子词 token:")
final_tokens = sorted(set(tok for word in vocab for tok in word.split()))
print(final_tokens)

## 第三章：字节级 BPE——处理任意文字的终极方案

标准 BPE 有一个根本缺陷：如果训练语料里没有某个字符（比如某个生僻字、新发明的 emoji），模型就无法处理它。

**GPT-2 开始采用的解决方案：先把文本转成字节（bytes），再在字节上运行 BPE。**

### 为什么字节是终极基础？

```
任意文本  --UTF-8编码-->  字节序列  (每个字节: 0~255 的整数)
```

- **字节只有 256 种**，是完备且有限的集合
- **任何文字都能用 UTF-8 字节表示**，不存在未知字符
- 在字节上运行 BPE 后，常见词的字节序列会自然合并成整体 token
- 罕见词、新词、emoji 都能被字节分解，不会崩溃

### 字节数规律（影响 token 消耗）

| 字符类型 | 示例 | UTF-8 字节数 |
|---------|------|------------|
| ASCII 英文 | `A`, `z`, `!` | **1 字节** |
| 中文 / 日文 / 韩文 | `你`, `日`, `한` | **3 字节** |
| emoji | `🎉`, `🤗`, `🚀` | **4 字节** |
| 扩展拉丁字母 | `é`, `ü`, `ñ` | **2 字节** |

这直接解释了为什么中文 token 数比英文多：同等字节数下，中文能表达更多信息，但每个汉字消耗的字节（和 token）更多。

In [ ]:
# 字节级编码展示
text_samples = [
    ("英文字母",  "Hello World!"),
    ("中文",      "你好世界"),
    ("Emoji",    "🎉🤗🚀"),
    ("代码",      "def f(x): return x*2"),
    ("混合",      "AI 人工智能 2024 🤖"),
]

print(f"{'类型':<10} {'文本':<22} {'字符数':>6} {'字节数':>6} {'字节/字符':>9}")
print("-" * 58)
for label, text in text_samples:
    byt   = text.encode('utf-8')
    ratio = len(byt) / len(text)
    print(f"{label:<10} {repr(text):<22} {len(text):>6} {len(byt):>6} {ratio:>9.2f}")

print("\n关键规律：")
print("  英文 1字符 = 1字节（向后兼容 ASCII）")
print("  中文 1字符 = 3字节（较大的 Unicode 码点）")
print("  emoji 1字符 = 4字节（辅助平面字符）")
print()
print("字节级 BPE 的效果：")
print("  常见英文词 -> 直接合并成整体 token（如 'the' = 1 token）")
print("  常见中文字 -> 同样会被合并（如 '人工' 可能是 1 token）")
print("  新词/emoji -> 退化成若干字节 token，始终可处理，不崩溃")

# 可视化字节分布
fig, axes = plt.subplots(len(text_samples), 1, figsize=(14, len(text_samples)*1.3 + 0.5))
for ax, (label, text) in zip(axes, text_samples):
    byt = list(text.encode('utf-8'))
    for x, b in enumerate(byt):
        color = plt.cm.plasma(b / 255)
        rect  = mpatches.FancyBboxPatch((x, 0.1), 0.85, 0.8,
                boxstyle="round,pad=0.05", facecolor=color, edgecolor='white', linewidth=0.7)
        ax.add_patch(rect)
        ax.text(x+0.425, 0.5, str(b), ha='center', va='center', fontsize=6.5,
                color='white' if b > 100 else 'black')
    max_bytes = max(len(t.encode()) for _, t in text_samples)
    ax.set_xlim(-0.2, max_bytes + 0.5)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_ylabel(f"{label}\n({len(byt)}字节)", rotation=0,
                  ha='right', va='center', fontsize=9, labelpad=55)
    ax.set_facecolor('#f5f5f5')
    if ax != axes[-1]:
        ax.set_xticks([])

axes[-1].set_xlabel('字节位置（每格 = 1 字节，格内数字 = 字节值 0~255）')
fig.suptitle('不同类型文本的 UTF-8 字节表示\n（字节级 BPE 在这一层面上进行合并）',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('byte_level_bpe.png', dpi=110, bbox_inches='tight')
plt.show()

## 第四章：tiktoken 实战

tiktoken 是 OpenAI 开源的高效分词库，支持 GPT 系列所有分词器。
安装：`pip install tiktoken`

**主要编码方案：**

| 编码名 | 使用模型 | 词表大小 |
|-------|---------|---------|
| `cl100k_base` | GPT-4, GPT-3.5-turbo | 100,277 |
| `o200k_base` | GPT-4o, GPT-4o mini | 200,019 |
| `p50k_base` | GPT-3 (text-davinci) | 50,281 |

> **注意：** Claude、LLaMA、Qwen 等模型使用各自独立的分词器，不能用 tiktoken 来估算它们的 token 数（差异可达 10%-30%）。

In [ ]:
import tiktoken

# 加载 GPT-4 的分词器
enc = tiktoken.get_encoding("cl100k_base")
print(f"词表大小: {enc.n_vocab:,}")

examples = [
    "Hello, world!",
    "你好，世界！",
    "The Transformer architecture changed NLP forever.",
    "def fibonacci(n): return n if n<=1 else fibonacci(n-1)+fibonacci(n-2)",
    "GPT-4o uses o200k_base tokenizer with 200K vocabulary.",
    "RLHF (Reinforcement Learning from Human Feedback) is key.",
]

print(f"\n{'文本 (截断60字)':<62} {'tokens':>7} {'字符':>6} {'tok/字符':>8}")
print("=" * 87)
for text in examples:
    toks = enc.encode(text)
    disp = repr(text[:58] + ('...' if len(text)>58 else ''))
    print(f"{disp:<62} {len(toks):>7} {len(text):>6} {len(toks)/len(text):>8.3f}")

print("\n" + "=" * 55)
print("分词详情（|token| 展示）:")
for text in examples[:4]:
    tokens  = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    display = " | ".join(repr(d) for d in decoded)
    print(f"\n  原文: {text!r}")
    print(f"  分词: {display}")
    print(f"  数量: {len(tokens)} tokens")

In [ ]:
# 中英文 token 效率深度对比
cn_text = "人工智能正在彻底改变各行各业，从医疗保健到金融服务，再到教育和科学研究。"
en_text = ("Artificial intelligence is fundamentally transforming every industry, "
           "from healthcare to financial services, education, and scientific research.")

cn_tokens = enc.encode(cn_text)
en_tokens = enc.encode(en_text)
cn_decoded = [enc.decode([t]) for t in cn_tokens]
en_decoded = [enc.decode([t]) for t in en_tokens]

print("中文分词结果：")
print(f"  原文 ({len(cn_text)}字符): {cn_text}")
print(f"  分词 ({len(cn_tokens)} tokens): " + " | ".join(repr(d) for d in cn_decoded))
print()
print("英文分词结果：")
print(f"  原文 ({len(en_text)}字符): {en_text[:60]}...")
print(f"  分词 ({len(en_tokens)} tokens): " + " | ".join(repr(d) for d in en_decoded[:12]) + "...")

print()
print("效率对比：")
print(f"  中文: {len(cn_tokens)/len(cn_text):.3f} tokens/字符")
print(f"  英文: {len(en_tokens)/len(en_text):.3f} tokens/字符")
ratio = (len(cn_tokens)/len(cn_text)) / (len(en_tokens)/len(en_text))
print(f"  中文消耗 token 约是英文的 {ratio:.1f} 倍（按字符数计）")
print()
print("历史演变：")
print("  GPT-3 (p50k): 中文每字符约 2-3 tokens（非常贵）")
print("  GPT-4 (cl100k): 中文每字符约 0.6-1 tokens（大幅优化）")
print("  LLaMA-3 (128K词表): 进一步优化，常见汉字大多是单 token")
print("  Qwen2.5 (150K词表): 专门针对中文调优，中文效率最高")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：token/字符数对比
x = np.arange(2)
labels = ['中文', '英文']
tok_c  = [len(cn_tokens), len(en_tokens)]
char_c = [len(cn_text), len(en_text)]
width  = 0.35
b1 = axes[0].bar(x - width/2, tok_c,  width, label='Token 数', color='#3498db', edgecolor='white')
b2 = axes[0].bar(x + width/2, char_c, width, label='字符数',   color='#e67e22', edgecolor='white')
for bar in list(b1) + list(b2):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 str(int(bar.get_height())), ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=13)
axes[0].set_ylabel('数量')
axes[0].set_title('同等语义的中英文\nToken 数 vs 字符数对比')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 右图：每个 token 的字符长度分布
cn_lens = [len(enc.decode([t])) for t in cn_tokens]
en_lens = [len(enc.decode([t])) for t in en_tokens]
max_len = max(max(cn_lens), max(en_lens))
bins    = range(1, max_len + 2)
axes[1].hist(cn_lens, bins=bins, alpha=0.7, label='中文 token', color='#3498db',
             density=True, align='left', rwidth=0.7)
axes[1].hist(en_lens, bins=bins, alpha=0.7, label='英文 token', color='#e74c3c',
             density=True, align='left', rwidth=0.7)
axes[1].set_xlabel('每个 token 包含的字符数')
axes[1].set_ylabel('占比')
axes[1].set_title('Token 长度分布\n（英文常见整词作为 1 token，所以分布更靠右）')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cn_en_token.png', dpi=120, bbox_inches='tight')
plt.show()

## 第五章：Token 数量与 API 成本

大多数 LLM API 按 token 计费，精确估算 token 数是控制成本的基础。

### 主流模型定价参考（2025 年）

| 模型 | 输入 /1M token | 输出 /1M token | 特点 |
|------|--------------|--------------|------|
| GPT-4o | $2.5 | $10 | 通用高质量 |
| GPT-4o mini | $0.15 | $0.6 | 极低成本 |
| Claude Sonnet 4 | $3 | $15 | 强推理，长上下文 |
| Claude Haiku 4 | $0.8 | $4 | 高并发低延迟 |

*价格持续变化，请以官方定价为准*

### 上下文窗口对比

| 模型 | 上下文窗口 | 约合汉字数 |
|------|-----------|---------|
| GPT-3.5 | 16K | ~12,000 字 |
| GPT-4o | 128K | ~96,000 字 |
| Claude 3.5 Sonnet | 200K | ~150,000 字 |
| Gemini 1.5 Pro | 1M | ~750,000 字（整部长篇小说）|

上下文窗口决定了模型的「记忆上限」，超出部分会被截断。这也是为什么需要 RAG 等长文档处理技术（下一章会讲）。

In [ ]:
def estimate_cost(input_text, output_tokens=500,
                  in_price=3.0, out_price=15.0,
                  model="Claude Sonnet", encoding="cl100k_base"):
    enc = tiktoken.get_encoding(encoding)
    in_toks   = len(enc.encode(input_text))
    in_cost   = in_toks / 1_000_000 * in_price
    out_cost  = output_tokens / 1_000_000 * out_price
    total_usd = in_cost + out_cost
    return {
        "model": model, "input_tokens": in_toks, "output_tokens": output_tokens,
        "in_cost": in_cost, "out_cost": out_cost,
        "total_usd": total_usd, "total_rmb": total_usd * 7.2,
    }

# 典型使用场景的成本
scenarios = [
    ("简单问答",  "你是助手。\n\n用户：什么是机器学习？",                        300),
    ("文档摘要",  "你是摘要专家。\n\n文档：" + "这是一段很长的文档内容。" * 300,  500),
    ("代码生成",  "你是Python专家，请为以下需求写完整代码：实现一个RESTful API服务",1200),
    ("多轮对话",  "用户：你好\n助手：你好！\n" * 20 + "用户：总结一下",          200),
]

print(f"{'场景':<10} {'输入tok':>8} {'输出tok':>8} {'合计$':>9} {'合计元':>9}")
print("-" * 50)
for name, text, out_toks in scenarios:
    r = estimate_cost(text, out_toks)
    print(f"{name:<10} {r['input_tokens']:>8,} {r['output_tokens']:>8,} "
          f"{r['total_usd']:>9.5f} {r['total_rmb']:>9.4f}")

print()
print("5 个节省成本的实用技巧：")
print("  1. Prompt Caching：重复 system prompt 可缓存，多轮对话显著省钱")
print("  2. 限制输出长度：明确要求「用3句话回答」避免模型啰嗦")
print("  3. 压缩对话历史：多轮后对早期轮次先摘要再放入 context")
print("  4. 合理选模型：简单任务用 Haiku/mini，复杂推理才用 Sonnet/4o")
print("  5. Batch API：批量请求通常有 50% 折扣（有延迟 SLA）")

In [ ]:
def check_context(text, limit=128_000, encoding="cl100k_base", model="GPT-4o"):
    enc   = tiktoken.get_encoding(encoding)
    toks  = len(enc.encode(text))
    safe  = int(limit * 0.9)
    over  = toks > safe
    rem   = safe - toks
    print(f"[{model}] 上下文检查")
    print(f"  文本字符数: {len(text):>10,}")
    print(f"  Token 数:   {toks:>10,}")
    print(f"  安全上限:   {safe:>10,}  (总限 {limit:,} 的 90%)")
    if over:
        print(f"  状态: 超出 {-rem:,} tokens！需要截断或 RAG 处理")
    else:
        print(f"  状态: 剩余 {rem:,} tokens 可用 (已用 {toks/safe*100:.1f}%)")
    return toks

print("场景1：普通问题")
check_context("请解释 Transformer 架构的核心原理。")

print()
print("场景2：超长文档")
long_doc = "这是一段文档内容，包含大量重要信息。" * 5000
check_context(long_doc)

## 第六章：词嵌入——从整数到语义向量

分词后，每个 token 是一个整数 ID（如 9527）。神经网络不能直接处理整数——整数大小 9527 > 42 在这里毫无意义。

**解决方案：用可训练的「查找表」把每个 ID 映射到高维实数向量。**

### 查找表的结构

```
词表大小 = 100,000
嵌入维度 = 4,096  (LLaMA-3 8B)

查找表形状 = (100,000, 4,096) = 4 亿个参数
```

对每个 token ID，取查找表对应的行作为其向量表示。这个查找表在训练过程中被优化。

### 为什么向量能表示语义？

训练后，语义相近的词会在向量空间中聚集，语义关系对应向量方向：

```
词向量运算示例（Word2Vec 发现的规律，现代 LLM 中更普遍）：
  vec("巴黎") - vec("法国") + vec("中国") ≈ vec("北京")    [首都-国家 关系]
  vec("国王") - vec("男人") + vec("女人") ≈ vec("女王")    [性别变换 方向]
  vec("跑") - vec("run") ≈ 零向量                          [中英文对应]
```

这些关系**不是人工编写的**，完全从大量文本中自动学到。

### 静态嵌入 vs 上下文嵌入

| 类型 | 方法 | 特点 |
|------|------|------|
| 静态嵌入 | Word2Vec, GloVe | 每个词固定一个向量，「苹果」永远是同一个向量 |
| 上下文嵌入 | BERT, GPT | 同一词在不同句子中向量不同（「苹果」在「吃苹果」和「苹果公司」中不同）|

现代 LLM 都是上下文嵌入，表达能力远强于静态嵌入。

In [ ]:
# 用 2D 向量演示嵌入的语义属性（真实是几千维）
words_2d = {
    'cat':     [0.8,  -0.6],
    'kitten':  [0.7,  -0.5],
    'dog':     [-0.7, -0.6],
    'puppy':   [-0.6, -0.5],
    'fish':    [0.0,  -0.9],
    'king':    [0.7,   0.8],
    'queen':   [-0.6,  0.8],
    'man':     [0.6,   0.5],
    'woman':   [-0.5,  0.5],
    'Paris':   [0.4,   0.2],
    'France':  [0.2,  -0.1],
    'Beijing': [-0.5,  0.2],
    'China':   [-0.7, -0.1],
}

groups_color = {
    'cat': '#27ae60', 'kitten': '#2ecc71', 'dog': '#27ae60',
    'puppy': '#2ecc71', 'fish': '#16a085',
    'king': '#e74c3c', 'queen': '#e74c3c',
    'man': '#c0392b', 'woman': '#c0392b',
    'Paris': '#3498db', 'France': '#2980b9',
    'Beijing': '#3498db', 'China': '#2980b9',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左图：2D 语义空间
for word, vec in words_2d.items():
    c = groups_color[word]
    axes[0].scatter(vec[0], vec[1], c=c, s=180, zorder=5, edgecolors='white', linewidth=1.5)
    axes[0].annotate(word, (vec[0], vec[1]), textcoords="offset points",
                     xytext=(6, 4), fontsize=10, fontweight='bold')

# 类比关系箭头
def draw_pair(ax, w1, w2, col='gray'):
    v1, v2 = words_2d[w1], words_2d[w2]
    ax.annotate('', xy=v2, xytext=v1,
                arrowprops=dict(arrowstyle='->', color=col, lw=1.5, linestyle='--'))

draw_pair(axes[0], 'king', 'queen', '#e74c3c')
draw_pair(axes[0], 'man', 'woman', '#c0392b')
draw_pair(axes[0], 'France', 'Paris', '#3498db')
draw_pair(axes[0], 'China', 'Beijing', '#2980b9')

axes[0].text(0.5, 0.92, 'king-queen 平行于 man-woman\n=> 性别方向',
             transform=axes[0].transAxes, ha='center', fontsize=9, color='#c0392b',
             bbox=dict(boxstyle='round', facecolor='#ffeaea', alpha=0.8))
axes[0].text(0.5, 0.82, 'France-Paris 平行于 China-Beijing\n=> 国家-首都方向',
             transform=axes[0].transAxes, ha='center', fontsize=9, color='#2980b9',
             bbox=dict(boxstyle='round', facecolor='#eaf3ff', alpha=0.8))

axes[0].axhline(0, color='k', linewidth=0.4, alpha=0.3)
axes[0].axvline(0, color='k', linewidth=0.4, alpha=0.3)
axes[0].set_title('词嵌入语义空间（2D 示意）\n语义相近的词聚集，语义关系对应向量方向', fontsize=11)
axes[0].grid(alpha=0.2)

# 右图：余弦相似度矩阵
selected = ['cat', 'dog', 'kitten', 'king', 'queen', 'man', 'woman', 'Paris', 'China']
n = len(selected)
sim = np.zeros((n, n))
for i, w1 in enumerate(selected):
    for j, w2 in enumerate(selected):
        v1, v2 = np.array(words_2d[w1]), np.array(words_2d[w2])
        sim[i,j] = np.dot(v1,v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))

im = axes[1].imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_xticks(range(n))
axes[1].set_yticks(range(n))
axes[1].set_xticklabels(selected, rotation=45, ha='right', fontsize=10)
axes[1].set_yticklabels(selected, fontsize=10)
axes[1].set_title('词语余弦相似度矩阵\n(绿=相似，红=不相似)', fontsize=11)
for i in range(n):
    for j in range(n):
        axes[1].text(j, i, f'{sim[i,j]:.1f}', ha='center', va='center',
                     fontsize=8, color='black' if abs(sim[i,j]) < 0.8 else 'white')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig('word_embeddings.png', dpi=110, bbox_inches='tight')
plt.show()

# 验证向量运算
k = np.array(words_2d['king'])
m = np.array(words_2d['man'])
w = np.array(words_2d['woman'])
q = np.array(words_2d['queen'])
pred = k - m + w
cos = np.dot(pred, q) / (np.linalg.norm(pred) * np.linalg.norm(q))
print(f"\nking - man + woman = {pred}")
print(f"queen              = {q}")
print(f"余弦相似度 = {cos:.4f}  (越接近 1 说明向量运算确实对应语义关系)")

In [ ]:
import torch
import torch.nn as nn

# nn.Embedding 本质上就是一个可训练的查找表
vocab_size = 50257   # GPT-2 词表大小
d_model    = 768     # GPT-2 Small 嵌入维度

emb_layer = nn.Embedding(vocab_size, d_model)
print(f"Embedding 查找表: {vocab_size:,} 个词 x {d_model} 维")
print(f"总参数量: {vocab_size * d_model:,}  ({vocab_size*d_model/117e6*100:.1f}% of GPT-2 Small)")
print()

# 演示查找操作
token_ids = torch.tensor([9527, 1234, 50256, 0])  # 4 个 token ID
with torch.no_grad():
    embeddings = emb_layer(token_ids)

print(f"输入 IDs: {token_ids.tolist()}")
print(f"输出向量 shape: {embeddings.shape}  (4个token, 每个{d_model}维)")
print(f"第一个 token 的前15维: {embeddings[0, :15].numpy().round(4)}")

# 各模型 Embedding 层参数量
print()
print(f"{'模型':<15} {'词表大小':<10} {'d_model':<8} {'Emb参数量':>12} {'占模型比':>8}")
print("-" * 60)
models = [
    ("GPT-2 Small",  50257, 768,   117e6),
    ("GPT-2 XL",     50257, 1600,  1542e6),
    ("GPT-3 (175B)", 50257, 12288, 175e9),
    ("LLaMA-3 8B",   128256, 4096, 8e9),
    ("Qwen2.5 7B",   151936, 3584, 7e9),
]
for name, vs, dm, total in models:
    ep  = vs * dm
    pct = ep / total * 100
    print(f"{name:<15} {vs:<10,} {dm:<8,} {ep:>12,} {pct:>7.1f}%")

## 本章总结

分词是连接「人类语言」和「神经网络数字」的桥梁。整条流水线：

```
原始文本
  |--(tiktoken/tokenizer)--> Token IDs  [整数序列]
  |--(Embedding 查表)------> 向量序列   [(seq_len, d_model) 的浮点矩阵]
  |--(Transformer Blocks)--> 上下文向量 [每个 token 知道了整个上下文]
  |--(Linear + Softmax)----> 概率分布   [预测下一个 token 是哪个]
```

| 知识点 | 核心结论 |
|-------|---------|
| 子词分词 | 平衡序列长度和词表大小，是目前最优方案 |
| BPE 算法 | 迭代合并高频相邻对，学出合并规则列表 |
| 字节级 BPE | 在 UTF-8 字节上运行，任意语言 emoji 均可处理 |
| 中文成本 | 同等信息量下消耗 token 更多，词表大的模型改善更好 |
| Embedding | 可训练查找表，将 ID 映射到语义向量空间 |
| 向量语义 | 语义相近的词向量距离近，语义关系对应向量方向 |
| 上下文窗口 | 模型处理上限，超出需要截断或 RAG |

**下一节：** Prompt Engineering——用同样的模型，通过设计提示词大幅提升输出质量。